# Phase 1: Baseline — DocLayout-YOLO-Indic

**Goal:** Reproduce the DocLayout-YOLO English baseline on D4LA, then measure the
zero-shot performance gap on Indic-script documents. This gap motivates Phase 2
(synthetic Indic pretraining + self-training).

**Run order:** Cell 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14 → 15

**Colab vs Cluster:**
- Cells 0-15 run fully on Colab (T4 GPU). Produces proxy metrics.
- Real mAP@[.5:.95] on full D4LA / DocLayNet / IndicDLP requires the cluster (June 3-8).

**Key fixes over previous version:**
- ✅ Single shared config cell — checkpoint filename defined once, used everywhere
- ✅ Correct D4LA checkpoint size validation (~40 MB is normal, NOT a nano model)
- ✅ Fixed Cell 7 & 8 wrong path (was DocLayNet, now correctly D4LA)
- ✅ Fixed Cell 5 / 13 / 15 hardcoded filename mismatch
- ✅ Cell descriptions updated with goal, dataset, and source details


## Cell 0 — Shared Config & Session Bootstrap
**Run this cell FIRST, every time you open a new Colab session.**

### What this cell does
Sets up the entire session: mounts Google Drive, installs all packages, defines the
checkpoint to use, and writes a tiny `.cfg` file so every subsequent cell can import
the correct filename without duplication.

### Checkpoint selection
| HF Token? | Repo | Filename | mAP D4LA | File size |
|-----------|------|----------|----------|-----------|
| No (public) | `juliozhao/DocLayout-YOLO-D4LA-from_scratch` | `doclayout_yolo_d4la_imgsz1600_from_scratch.pt` | 69.8% | ~40 MB |
| Yes | `juliozhao/DocLayout-YOLO-D4LA-Docsynth300K_pretrained` | `doclayout_yolo_d4la_imgsz1600_docsynth_pretrain.pt` | 70.3% | ~40 MB |

> **Note:** Both D4LA checkpoints are legitimately ~40 MB — this is NOT a nano model.
> The larger (~170 MB) `DocLayout-YOLO-DocStructBench` is a *different model* (general
> document prediction, not D4LA-trained). We use the D4LA-trained checkpoints here
> because Phase 1 goal is to reproduce the paper's D4LA numbers.

### All repos / datasets referenced in this notebook
| Resource | HuggingFace URL | Size | Purpose |
|----------|----------------|------|---------|
| D4LA-from_scratch checkpoint | `juliozhao/DocLayout-YOLO-D4LA-from_scratch` | ~40 MB | Phase 1 baseline model (public) |
| D4LA-Docsynth300K checkpoint | `juliozhao/DocLayout-YOLO-D4LA-Docsynth300K_pretrained` | ~40 MB | Phase 1 baseline model (token) |
| D4LA dataset (YOLO format) | `juliozhao/doclayout-yolo-D4LA` (dataset) | 1.28 GB tar | English baseline eval data |
| DocLayout-YOLO code | `github.com/opendatalab/DocLayout-YOLO` | — | Model architecture |
| Wikipedia REST API | `{lang}.wikipedia.org/api/rest_v1/page/pdf/{title}` | ~1-3 MB/article | Indic proxy pages |

⚠️ **Before running:** Runtime → Change runtime type → T4 GPU → Save


In [ ]:
# ── Cell 0: Shared Config & Session Bootstrap ──
# ⚠️  Runtime → Change runtime type → T4 GPU → Save  BEFORE running this cell

import os, sys, subprocess, json
from pathlib import Path
from google.colab import drive

# ── 1. Mount Drive ──
drive.mount('/content/drive', force_remount=True)

# ── 2. Project root ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# ── 3. Install packages ──
print("Installing packages...")
subprocess.run([
    'pip', 'install', '-q',
    'ultralytics>=8.0.0',
    'huggingface_hub',
    'pycocotools',
    'opencv-python',
    'matplotlib',
    'tqdm',
    'gdown',
    'pymupdf',
    'requests',
], check=True)
print("  packages installed ✓")

# ── 4. GPU check ──
import torch
print(f"\nPyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — Runtime → Change runtime type → T4 GPU → Save")

# ── 5. Checkpoint selection ──
# Set your HuggingFace token below if you have one (gives +0.5% mAP, same model size).
# Leave as empty string '' to use the fully public checkpoint — both work fine.
HF_TOKEN = os.environ.get('HF_TOKEN', '')  # or paste token string here

if HF_TOKEN:
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-Docsynth300K_pretrained'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_docsynth_pretrain.pt'
    CKPT_MAP      = 70.3
    CKPT_NOTE     = 'D4LA + DocSynth300K pretrained  (requires HF token)  — paper best'
else:
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch  (fully public, no token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

# ── 6. Save config so all other cells can load it without repeating the logic ──
cfg_dir = PROJECT_ROOT / 'output' / 'checkpoints'
cfg_dir.mkdir(parents=True, exist_ok=True)
cfg = {
    'repo_id':  CKPT_REPO_ID,
    'filename': CKPT_FILENAME,
    'map':      CKPT_MAP,
    'note':     CKPT_NOTE,
    'hf_token': HF_TOKEN,
}
(cfg_dir / '.cfg').write_text(json.dumps(cfg, indent=2))

# ── 7. Print summary ──
print(f"\n{'='*65}")
print(f"  PROJECT ROOT : {PROJECT_ROOT}")
print(f"  CHECKPOINT   : {CKPT_FILENAME}")
print(f"  REPO         : {CKPT_REPO_ID}")
print(f"  PAPER mAP    : {CKPT_MAP}%  ({CKPT_NOTE})")
print(f"  FILE SIZE    : ~40 MB  (D4LA checkpoints are 40 MB — this is correct)")
print(f"  HF TOKEN     : {'set ✓' if HF_TOKEN else 'not set — using public repo'}")
print(f"{'='*65}")
print("\n✅ Cell 0 complete — proceed to Cell 1")


## Cell 1 — Create Directory Structure
**Goal:** Create the project folder tree on Google Drive so all subsequent cells
have a consistent place to read/write data.

No downloads happen here. This is idempotent — safe to re-run.


In [ ]:
# ── Cell 1: Create Directory Structure ──
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

dirs = [
    'data/raw/D4LA/images/test',
    'data/raw/D4LA/labels/test',
    'data/raw/DocLayNet',
    'data/raw/IndicDLP/images/test',
    'output/evaluation',
    'output/checkpoints',
    'output/logs',
    'src',
]
for d in dirs:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

print("Directory tree created:")
for d in dirs:
    print(f"  ✓  {PROJECT_ROOT / d}")
print("\n✅ Cell 1 complete")


## Cell 2 — Clone DocLayout-YOLO Repository
**Goal:** Clone the official DocLayout-YOLO repo and install it as a Python package.
This provides the `doclayout_yolo` module and `YOLOv10` model class used in every
inference cell.

**Source:** `github.com/opendatalab/DocLayout-YOLO`  
**Paper:** Zhao et al. (2024), arXiv 2410.12628  
**Cloned to:** `/content/DocLayout-YOLO` (local Colab disk, not Drive — re-clone each session)


In [ ]:
# ── Cell 2: Clone DocLayout-YOLO Repository ──
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

import subprocess
repo_dir = Path('/content/DocLayout-YOLO')

if not repo_dir.exists():
    print("Cloning DocLayout-YOLO repository...")
    subprocess.run([
        'git', 'clone',
        'https://github.com/opendatalab/DocLayout-YOLO.git',
        str(repo_dir)
    ], check=True)
    subprocess.run(['pip', 'install', '-q', '-e', str(repo_dir)], check=True)
    print("  Repo cloned and installed ✓")
else:
    print("Repo already exists — pulling latest...")
    subprocess.run(['git', '-C', str(repo_dir), 'pull'])
    subprocess.run(['pip', 'install', '-q', '-e', str(repo_dir)], check=True)

# Verify import
from doclayout_yolo import YOLOv10
print(f"\nRepo location : {repo_dir}")
print(f"doclayout_yolo import : ✓")
print("\n✅ Cell 2 complete")


## Cell 3 — Download D4LA-Trained Checkpoint
**Goal:** Download the correct DocLayout-YOLO checkpoint trained on D4LA — the
English document layout benchmark. This is the model we will evaluate as our
"English baseline" and then compare against Indic-script pages in Cell 13.

### Which checkpoint and why?
We use the **D4LA-trained** checkpoints (not DocStructBench) because:
- Phase 1 goal is to reproduce D4LA paper numbers (69.8% / 70.3% mAP)
- The DocStructBench model (~170 MB) is trained for general prediction across
  diverse doc types — it has different class labels and cannot be fairly compared
  to D4LA ground-truth annotations

### Checkpoint details
| Property | from_scratch (public) | Docsynth300K_pretrained (token) |
|----------|-----------------------|----------------------------------|
| HF Repo | `juliozhao/DocLayout-YOLO-D4LA-from_scratch` | `juliozhao/DocLayout-YOLO-D4LA-Docsynth300K_pretrained` |
| Filename | `doclayout_yolo_d4la_imgsz1600_from_scratch.pt` | `doclayout_yolo_d4la_imgsz1600_docsynth_pretrain.pt` |
| File size | **~40 MB** ✓ (correct, not nano) | **~40 MB** ✓ |
| Classes | 27 D4LA classes | 27 D4LA classes |
| Image size | 1600 px | 1600 px |
| mAP@[.5:.95] | 69.8% | 70.3% |
| AP50 | 81.7% | 82.4% |
| Token needed | No | Yes |

The checkpoint is cached to Google Drive so it survives session restarts.


In [ ]:
# ── Cell 3: Download D4LA Checkpoint ──
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

import subprocess, json
from huggingface_hub import hf_hub_download

# Load HF token from config if set
_cfg = json.loads((PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg').read_text())
HF_TOKEN = _cfg.get('hf_token', '')

print(f"Checkpoint : {CKPT_FILENAME}")
print(f"Repo       : {CKPT_REPO_ID}")
print(f"Paper mAP  : {CKPT_MAP}%")
print(f"File size  : ~40 MB  (this is correct for D4LA checkpoints)")
print()

# ── Restore from Drive cache if available ──
if not CKPT_LOCAL.exists() and CKPT_DRIVE.exists():
    size_mb = CKPT_DRIVE.stat().st_size / 1e6
    print(f"Restoring from Drive ({size_mb:.0f} MB)...")
    shutil.copy(CKPT_DRIVE, CKPT_LOCAL)

if CKPT_LOCAL.exists():
    size_mb = CKPT_LOCAL.stat().st_size / 1e6
    print(f"Checkpoint ready : {CKPT_LOCAL.name}  ({size_mb:.0f} MB) ✓")
else:
    print(f"Downloading {CKPT_FILENAME} from HuggingFace...")
    kwargs = {'token': HF_TOKEN} if HF_TOKEN else {}
    local_path = hf_hub_download(
        repo_id   = CKPT_REPO_ID,
        filename  = CKPT_FILENAME,
        local_dir = '/content',
        **kwargs,
    )
    if Path(local_path) != CKPT_LOCAL:
        shutil.copy(local_path, CKPT_LOCAL)
    print(f"  Downloaded : {CKPT_LOCAL.name}  ({CKPT_LOCAL.stat().st_size/1e6:.0f} MB)")
    shutil.copy(CKPT_LOCAL, CKPT_DRIVE)
    print(f"  Cached to Drive ✓")

# ── Verify: load model and confirm D4LA class alignment ──
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)
from doclayout_yolo import YOLOv10

model   = YOLOv10(str(CKPT_LOCAL))
classes = list(model.names.values())
name_ok = len(classes) == 27 and classes[0] == 'DocTitle'

print(f"\nClasses ({len(classes)}) : {classes}")
print(f"\nClass 0     : '{classes[0]}'  (expected 'DocTitle')")
print(f"Num classes : {len(classes)}     (expected 27)")
print(f"Alignment   : {'✅ ALIGNED — D4LA labels match' if name_ok else '❌ MISMATCH — wrong checkpoint'}")

# ── Cross-check against D4LA label files if already downloaded ──
LBL_DIR = PROJECT_ROOT / 'data' / 'raw' / 'D4LA' / 'labels' / 'test'
label_files = list(LBL_DIR.glob('*.txt'))
if label_files:
    sample_ids = set()
    for lf in label_files[:10]:
        for line in lf.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts:
                sample_ids.add(int(parts[0]))
    print(f"\nGT class IDs found in D4LA labels : {sorted(sample_ids)}")
    all_ok = all(cid < len(classes) for cid in sample_ids)
    print(f"Full alignment check : {'✅ ALL IDs IN RANGE' if all_ok else '❌ SOME IDs OUT OF RANGE'}")
else:
    print("\n(D4LA labels not yet downloaded — run Cell 5 first, then optionally re-run this cell)")

print("\n✅ Cell 3 complete")


## Cell 4 — Quick Model Sanity Check
**Goal:** Verify that the downloaded checkpoint loads correctly and produces detections
on a synthetic document page. This confirms the model + GPU pipeline works before
we run on real data.

The test image is a programmatically generated page with coloured rectangles
representing: title block, two text columns, a figure, a table, and a caption.
We expect the model to detect 3–6 of these regions at conf ≥ 0.15.


In [ ]:
# ── Cell 4: Quick Model Sanity Check ──
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

import subprocess
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)

from doclayout_yolo import YOLOv10
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch

if not CKPT_LOCAL.exists() and CKPT_DRIVE.exists():
    shutil.copy(CKPT_DRIVE, CKPT_LOCAL)

if not CKPT_LOCAL.exists():
    print("❌ Checkpoint not found — run Cell 3 first.")
else:
    size_mb = CKPT_LOCAL.stat().st_size / 1e6
    print(f"Checkpoint : {CKPT_LOCAL.name}")
    print(f"Size       : {size_mb:.0f} MB")
    print(f"Note       : D4LA checkpoints are ~40 MB — this is correct (not nano)")

    model = YOLOv10(str(CKPT_LOCAL))
    print(f"Task       : {model.task}")
    print(f"Classes    : {model.model.nc} → {list(model.names.values())}")
    print(f"Device     : {'GPU ✓' if torch.cuda.is_available() else 'CPU ⚠️ slow'}")

    # ── Synthetic test page ──
    img = Image.new('RGB', (850, 1100), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    draw.rectangle([40,  25, 810,  85],  fill=(200, 220, 255))   # title block
    draw.rectangle([40, 105, 410, 680],  fill=(238, 238, 238))   # text col 1
    draw.rectangle([440,105, 810, 680],  fill=(238, 238, 238))   # text col 2
    draw.rectangle([40, 700, 400, 950],  fill=(220, 235, 220))   # figure
    draw.rectangle([420,700, 810, 950],  fill=(255, 240, 210))   # table
    draw.rectangle([40, 960, 810,1000],  fill=(245, 245, 245))   # caption
    test_path = '/content/test_doc.png'
    img.save(test_path)

    results = model.predict(source=test_path, imgsz=1024, conf=0.15, verbose=False)
    boxes   = results[0].boxes
    print(f"\nDetected {len(boxes)} regions on synthetic test page:")
    for b in boxes:
        print(f"  {model.names[int(b.cls[0])]:20s}  conf={float(b.conf[0]):.2f}")

    # ── Visualise ──
    COLORS = ['red','blue','green','orange','purple','brown','pink','gray','olive','cyan']
    fig, ax = plt.subplots(figsize=(6, 7))
    ax.imshow(img)
    for b in boxes:
        x1,y1,x2,y2 = b.xyxy[0].tolist()
        cid = int(b.cls[0])
        c = COLORS[cid % len(COLORS)]
        ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1, lw=2, edgecolor=c, facecolor='none'))
        ax.text(x1, y1-4, f"{model.names[cid]}:{float(b.conf[0]):.2f}",
                color=c, fontsize=7, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.6, pad=1))
    ax.axis('off')
    plt.title(f"Sanity check — {len(boxes)} detections on synthetic page", fontsize=10)
    plt.tight_layout()
    plt.savefig('/content/sanity_check.png', dpi=100)
    plt.show()

    if len(boxes) > 0:
        print("\n✅ Cell 4 PASSED — model loads and detects correctly")
    else:
        print("\n⚠️  No detections — lower conf threshold or check checkpoint (Cell 3)")


## Cell 5 — Download D4LA Dataset
**Goal:** Download the official D4LA dataset in YOLO format from HuggingFace and
extract a stratified sample for Colab evaluation.

**Dataset:** `juliozhao/doclayout-yolo-D4LA` (HuggingFace dataset repo)  
**Format:** YOLO format — `images/` + `labels/*.txt` (class_id cx cy w h per line)  
**Full size:** 1.28 GB tar → ~30,500 test images with annotations  
**Classes:** 27 D4LA classes (DocTitle, ParaTitle, ParaText, Table, Figure, ...)  
**Colab sample:** 60 images (stratified random from test split) — enough for P/R/F1  
**Paper reference:** Wang et al. 2024, same split used for 70.3% mAP result  

The tar is cached to Google Drive so re-runs skip the 5-10 min download.  
Full 30,500-image evaluation runs on the cluster (June 3-8) for real mAP@[.5:.95].


In [ ]:
# ── Cell 5: Download D4LA Dataset ──
# Source  : juliozhao/doclayout-yolo-D4LA  (HuggingFace dataset)
# Content : D4LA.tar.gz → images/ + labels/ + test.txt + train.txt
# Format  : YOLO format (images are JPG, labels are .txt per image)
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

import tarfile, random, subprocess
from huggingface_hub import hf_hub_download

D4LA_DIR  = PROJECT_ROOT / 'data' / 'raw' / 'D4LA'
IMG_DIR   = D4LA_DIR / 'images' / 'test'
LBL_DIR   = D4LA_DIR / 'labels' / 'test'
IMG_DIR.mkdir(parents=True, exist_ok=True)
LBL_DIR.mkdir(parents=True, exist_ok=True)

TAR_DRIVE = D4LA_DIR / 'D4LA.tar.gz'
TAR_LOCAL = Path('/content/D4LA.tar.gz')
MAX_PAGES = 60   # sample cap for Colab; cluster uses all ~30,500 images

existing_imgs = list(IMG_DIR.glob('*.jpg')) + list(IMG_DIR.glob('*.png'))
if len(existing_imgs) >= MAX_PAGES:
    print(f"D4LA already extracted ({len(existing_imgs)} images) ✓  — skipping download")
else:
    # ── Restore from Drive cache ──
    if TAR_DRIVE.exists() and TAR_DRIVE.stat().st_size > 1e8:
        print(f"Restoring D4LA tar from Drive ({TAR_DRIVE.stat().st_size/1e9:.2f} GB)...")
        shutil.copy(TAR_DRIVE, TAR_LOCAL)

    if not TAR_LOCAL.exists() or TAR_LOCAL.stat().st_size < 1e8:
        print("Downloading D4LA.tar.gz from HuggingFace (~1.28 GB, 5-10 min)...")
        local_path = hf_hub_download(
            repo_id   = 'juliozhao/doclayout-yolo-D4LA',
            repo_type = 'dataset',
            filename  = 'D4LA.tar.gz',
            local_dir = '/content',
        )
        if Path(local_path) != TAR_LOCAL:
            shutil.copy(local_path, TAR_LOCAL)
        print(f"  Downloaded: {TAR_LOCAL.stat().st_size/1e9:.2f} GB ✓")
        shutil.copy(TAR_LOCAL, TAR_DRIVE)
        print(f"  Cached to Drive ✓")

    # ── Extract test-split sample ──
    print(f"\nExtracting up to {MAX_PAGES} test images + labels...")
    test_stems, img_members, lbl_members = set(), {}, {}

    with tarfile.open(str(TAR_LOCAL), 'r:gz') as tar:
        all_members = tar.getmembers()

        for m in all_members:
            if m.name.endswith('test.txt') and m.isfile():
                for line in tar.extractfile(m).read().decode().strip().splitlines():
                    test_stems.add(Path(line.strip()).stem)
                print(f"  test.txt: {len(test_stems)} official test images")
                break

        for m in all_members:
            if not m.isfile():
                continue
            p = Path(m.name)
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                img_members[p.stem] = m
            elif p.suffix == '.txt' and 'label' in m.name.lower():
                lbl_members[p.stem] = m

        candidates = [s for s in (test_stems if test_stems else img_members.keys())
                      if s in img_members]
        random.seed(42)
        random.shuffle(candidates)
        sample = candidates[:MAX_PAGES]

        n_img = n_lbl = 0
        for stem in sample:
            data = tar.extractfile(img_members[stem])
            if data:
                (IMG_DIR / Path(img_members[stem].name).name).write_bytes(data.read())
                n_img += 1
            if stem in lbl_members:
                data = tar.extractfile(lbl_members[stem])
                if data:
                    (LBL_DIR / f"{stem}.txt").write_bytes(data.read())
                    n_lbl += 1

    print(f"  Images   : {n_img}")
    print(f"  Labels   : {n_lbl}  ← YOLO-format annotations ✓")

imgs   = list(IMG_DIR.glob('*.jpg')) + list(IMG_DIR.glob('*.png'))
labels = list(LBL_DIR.glob('*.txt'))
print(f"\n{'='*55}")
print(f"  D4LA ready")
print(f"  Images  : {len(imgs)}  →  {IMG_DIR}")
print(f"  Labels  : {len(labels)}  →  {LBL_DIR}")
print(f"  Repo    : juliozhao/doclayout-yolo-D4LA (HuggingFace dataset)")
print(f"  Full set: ~30,500 test images — cluster needed for full mAP")
print(f"{'='*55}")
if imgs:
    from PIL import Image as PILImage
    sample_img = PILImage.open(str(imgs[0]))
    print(f"\nSample image size : {sample_img.size[0]} x {sample_img.size[1]} px")
print("\n✅ Cell 5 complete — proceed to Cell 6 (inspect) or Cell 8 (evaluate)")


## Cell 6 — Inspect D4LA Test Images + Ground Truth Overlay
**Goal:** Visually confirm the downloaded D4LA images and their YOLO-format
annotations look correct before running evaluation.

Shows 4 sample pages with ground-truth boxes overlaid (coloured by class).
If labels are missing, shows predictions-only as a fallback.


In [ ]:
# ── Cell 6: Inspect D4LA Test Images + Ground Truth ──
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import subprocess

subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)
from doclayout_yolo import YOLOv10

IMG_DIR = PROJECT_ROOT / 'data' / 'raw' / 'D4LA' / 'images' / 'test'
LBL_DIR = PROJECT_ROOT / 'data' / 'raw' / 'D4LA' / 'labels' / 'test'
imgs    = sorted(list(IMG_DIR.glob('*.jpg')) + list(IMG_DIR.glob('*.png')))

if not imgs:
    print("No D4LA images found — run Cell 5 first.")
else:
    print(f"Found {len(imgs)} D4LA test images")
    model  = YOLOv10(str(CKPT_LOCAL)) if CKPT_LOCAL.exists() else None
    COLORS = ['red','blue','green','orange','purple','brown','pink','gray','olive','cyan',
              'lime','magenta','teal','navy','coral','gold','violet']

    def load_gt(lbl_path, w, h):
        boxes = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                p = line.strip().split()
                if len(p) < 5: continue
                cid = int(p[0]); cx,cy,bw,bh = map(float,p[1:5])
                x1=(cx-bw/2)*w; y1=(cy-bh/2)*h; x2=(cx+bw/2)*w; y2=(cy+bh/2)*h
                boxes.append((cid,x1,y1,x2,y2))
        return boxes

    n = min(4, len(imgs))
    fig, axes = plt.subplots(1, n, figsize=(5*n, 8))
    if n == 1: axes = [axes]

    for ax, p in zip(axes, imgs[:n]):
        img_obj = Image.open(str(p))
        w, h = img_obj.size
        ax.imshow(img_obj)
        gt = load_gt(LBL_DIR / (p.stem + '.txt'), w, h)
        cnames = list(model.names.values()) if model else []
        for (cid, x1, y1, x2, y2) in gt:
            c = COLORS[cid % len(COLORS)]
            ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,
                lw=2, edgecolor=c, facecolor=c, alpha=0.12))
            ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,
                lw=2, edgecolor=c, facecolor='none'))
            label = cnames[cid][:10] if cid < len(cnames) else f'cls{cid}'
            ax.text(x1, max(y1-4,0), label, color=c, fontsize=6, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.6, pad=0))
        gt_label = f"n_gt={len(gt)}" if gt else "no labels"
        ax.set_title(f"{p.stem[:20]}\n{gt_label}", fontsize=7)
        ax.axis('off')

    plt.suptitle("D4LA Test Pages — Ground Truth Annotations (coloured by class)", fontsize=11)
    plt.tight_layout()
    out = PROJECT_ROOT / 'output' / 'evaluation' / 'd4la_gt_samples.png'
    plt.savefig(str(out), dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Preview saved: {out}")
    print("\n✅ Cell 6 complete")


## Cell 7 — Confirm D4LA Images Ready
**Goal:** Quick sanity check — print image count, sample size, and label coverage
before running the full evaluation in Cell 8.


In [ ]:
# ── Cell 7: Confirm D4LA Images Ready ──
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

from PIL import Image

IMG_DIR = PROJECT_ROOT / 'data' / 'raw' / 'D4LA' / 'images' / 'test'
LBL_DIR = PROJECT_ROOT / 'data' / 'raw' / 'D4LA' / 'labels' / 'test'

imgs   = list(IMG_DIR.glob('*.jpg')) + list(IMG_DIR.glob('*.png'))
labels = list(LBL_DIR.glob('*.txt'))

print(f"D4LA images  : {len(imgs)}")
print(f"D4LA labels  : {len(labels)}")

if imgs:
    sample = Image.open(str(imgs[0]))
    print(f"Sample size  : {sample.size[0]} x {sample.size[1]} px")
    coverage = len(labels) / len(imgs) * 100 if imgs else 0
    print(f"Label cover  : {coverage:.0f}%")
    if coverage < 90:
        print("  ⚠️  Low label coverage — re-run Cell 5 to re-extract labels")
    else:
        print("  ✅ Ready for evaluation (Cell 8)")
else:
    print("❌ No images — run Cell 5 first.")


## Cell 8 — D4LA English Baseline Evaluation ★ CRITICAL
**Goal:** Run the DocLayout-YOLO model on the 60-image D4LA sample and compute
per-class Precision / Recall / F1 at IoU ≥ 0.5 using the YOLO-format ground-truth
annotations. This is the **English baseline** all Indic results are compared against.

**Metrics computed:**
- Per-class TP, FP, FN → Precision, Recall, F1 at IoU ≥ 0.5
- Macro-averaged P / R / F1 across all classes present in the sample
- Avg confidence per page (proxy metric used for Indic gap comparison in Cell 12)

**Paper benchmark:** mAP@[.5:.95] = 70.3% on full 30,500-image test set (Wang et al. 2024)  
Our Colab run is on 60 images → F1 ≈ macro proxy, not official mAP.  
Full mAP run on cluster (June 3-8) using all images.

**Results saved to:** `output/evaluation/phase1_english_baseline.json`


In [ ]:
# ── Cell 8: D4LA English Baseline Evaluation ★ CRITICAL ──
# Dataset  : D4LA test sample (juliozhao/doclayout-yolo-D4LA, 60 images)
# Labels   : YOLO-format GT from Cell 5
# Metric   : P/R/F1 at IoU≥0.5  +  avg confidence (proxy for Cell 12 gap)
# Paper    : mAP@[.5:.95] = 70.3%  (full 30,500 images — cluster needed)
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

import subprocess, json
from collections import defaultdict
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch

subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)
from doclayout_yolo import YOLOv10

IMG_DIR = PROJECT_ROOT / 'data' / 'raw' / 'D4LA' / 'images' / 'test'
LBL_DIR = PROJECT_ROOT / 'data' / 'raw' / 'D4LA' / 'labels' / 'test'

if not CKPT_LOCAL.exists() and CKPT_DRIVE.exists():
    shutil.copy(CKPT_DRIVE, CKPT_LOCAL)

imgs = sorted(list(IMG_DIR.glob('*.jpg')) + list(IMG_DIR.glob('*.png')))

if not imgs:
    print(f"❌ No D4LA images in {IMG_DIR}\n   Run Cell 5 first.")
elif not CKPT_LOCAL.exists():
    print("❌ Checkpoint missing — run Cell 3 first.")
else:
    model       = YOLOv10(str(CKPT_LOCAL))
    CLASS_NAMES = list(model.names.values())
    NUM_CLASSES = len(CLASS_NAMES)
    labels_found = list(LBL_DIR.glob('*.txt'))

    print(f"Device     : {'GPU ✓' if torch.cuda.is_available() else 'CPU ⚠️'}")
    print(f"Checkpoint : {CKPT_LOCAL.name}  ({CKPT_LOCAL.stat().st_size/1e6:.0f} MB)")
    print(f"Classes    : {NUM_CLASSES}  → {CLASS_NAMES}")
    print(f"Images     : {len(imgs)}")
    print(f"GT labels  : {len(labels_found)}")
    if not labels_found:
        print("⚠️  No labels found — re-run Cell 5 to ensure labels are extracted")

    def iou_xyxy(b1, b2):
        ix1=max(b1[0],b2[0]); iy1=max(b1[1],b2[1])
        ix2=min(b1[2],b2[2]); iy2=min(b1[3],b2[3])
        inter=max(0,ix2-ix1)*max(0,iy2-iy1)
        union=((b1[2]-b1[0])*(b1[3]-b1[1])+(b2[2]-b2[0])*(b2[3]-b2[1])-inter)
        return inter/union if union>0 else 0.0

    def load_gt(lbl_path, w, h):
        gt=[]
        if not lbl_path.exists(): return gt
        for line in lbl_path.read_text().strip().splitlines():
            parts=line.strip().split()
            if len(parts)<5: continue
            cid=int(parts[0]); cx,cy,bw,bh=map(float,parts[1:5])
            x1=(cx-bw/2)*w; y1=(cy-bh/2)*h; x2=(cx+bw/2)*w; y2=(cy+bh/2)*h
            gt.append((cid,x1,y1,x2,y2))
        return gt

    IOU_THRESH = 0.5
    per_class_tp = defaultdict(int); per_class_fp = defaultdict(int)
    per_class_fn = defaultdict(int); per_class_gt  = defaultdict(int)
    results_log, raw_preds = [], []

    print(f"\nRunning inference on {len(imgs)} D4LA pages (conf≥0.25, iou=0.45)...")
    print("─" * 60)

    for img_path in imgs:
        img_obj      = Image.open(str(img_path))
        img_w, img_h = img_obj.size
        lbl_path     = LBL_DIR / (img_path.stem + '.txt')
        gt_boxes     = load_gt(lbl_path, img_w, img_h)
        for (cid,*_) in gt_boxes: per_class_gt[cid] += 1

        res   = model.predict(source=str(img_path), imgsz=1024,
                               conf=0.25, iou=0.45, verbose=False)[0]
        preds = [(int(b.cls[0]), float(b.conf[0]), *b.xyxy[0].tolist()) for b in res.boxes]
        confs = [p[1] for p in preds]

        gt_matched = [False]*len(gt_boxes)
        for (pcid,pconf,px1,py1,px2,py2) in sorted(preds,key=lambda x:-x[1]):
            best_iou, best_j = 0.0, -1
            for j,(gcid,gx1,gy1,gx2,gy2) in enumerate(gt_boxes):
                if gt_matched[j] or gcid!=pcid: continue
                iou=iou_xyxy([px1,py1,px2,py2],[gx1,gy1,gx2,gy2])
                if iou>best_iou: best_iou,best_j=iou,j
            if best_iou>=IOU_THRESH and best_j>=0:
                per_class_tp[pcid]+=1; gt_matched[best_j]=True
            else:
                per_class_fp[pcid]+=1
        for j,(gcid,*_) in enumerate(gt_boxes):
            if not gt_matched[j]: per_class_fn[gcid]+=1

        results_log.append({
            "image":img_path.name,"num_detections":len(preds),
            "num_gt_boxes":len(gt_boxes),
            "avg_conf":round(sum(confs)/len(confs),3) if confs else 0.0,
            "has_labels":lbl_path.exists(),
        })
        raw_preds.append((img_path,res,gt_boxes,img_w,img_h))

    # ── Per-class metrics ──
    print(f"\n{'='*68}")
    print("D4LA BASELINE — Per-class Precision / Recall / F1")
    print(f"IoU≥{IOU_THRESH}  conf≥0.25  |  {len(imgs)} test images sampled from 30,500 total")
    print(f"{'='*68}")
    print(f"  {'Class':<22} {'GT':>5} {'TP':>5} {'FP':>5} {'FN':>5} {'Prec':>7} {'Rec':>7} {'F1':>7}")
    print("  "+"─"*64)

    class_metrics = {}
    all_cids = sorted(set(list(per_class_gt)+list(per_class_tp)+list(per_class_fp)))
    for cid in all_cids:
        tp=per_class_tp[cid]; fp=per_class_fp[cid]
        fn=per_class_fn[cid]; gt_n=per_class_gt[cid]
        prec=tp/(tp+fp) if tp+fp>0 else 0.0
        rec =tp/(tp+fn) if tp+fn>0 else 0.0
        f1  =2*prec*rec/(prec+rec) if prec+rec>0 else 0.0
        cname=CLASS_NAMES[cid] if cid<NUM_CLASSES else f'cls_{cid}'
        class_metrics[cname]={"gt":gt_n,"tp":tp,"fp":fp,"fn":fn,
            "precision":round(prec,3),"recall":round(rec,3),"f1":round(f1,3)}
        flag="✅" if f1>=0.5 else ("⚠️" if f1>=0.3 else "❌")
        print(f"  {cname:<22} {gt_n:>5} {tp:>5} {fp:>5} {fn:>5} {prec:>7.1%} {rec:>7.1%} {f1:>7.3f}  {flag}")

    cls_with_gt = [m for m in class_metrics.values() if m['gt']>0]
    macro_prec = sum(m['precision'] for m in cls_with_gt)/len(cls_with_gt)
    macro_rec  = sum(m['recall']    for m in cls_with_gt)/len(cls_with_gt)
    macro_f1   = sum(m['f1']        for m in cls_with_gt)/len(cls_with_gt)
    avg_det    = sum(r['num_detections'] for r in results_log)/len(results_log)
    avg_conf   = sum(r['avg_conf']       for r in results_log)/len(results_log)
    labeled_pct= sum(1 for r in results_log if r['has_labels'])/len(results_log)*100
    ok = avg_det>=3.0 and avg_conf>=0.5

    print("  "+"─"*64)
    print(f"  {'MACRO AVG':<22} {'':>5} {'':>5} {'':>5} {'':>5} {macro_prec:>7.1%} {macro_rec:>7.1%} {macro_f1:>7.3f}")
    print(f"\n  Pages evaluated      : {len(results_log)}")
    print(f"  Pages with GT labels : {labeled_pct:.0f}%")
    print(f"  Avg detections/page  : {avg_det:.1f}")
    print(f"  Avg confidence       : {avg_conf:.1%}  ← proxy metric for Cell 12 gap")
    print(f"  Macro F1 @ IoU≥0.5   : {macro_f1:.3f}")
    print(f"\n  Paper benchmark: mAP@[.5:.95] = {CKPT_MAP}% on 30,500 images (Wang et al. 2024)")
    print(f"  {'✅ BASELINE CONFIRMED' if ok else '⚠️  Low results — check checkpoint (Cell 3)'}")
    print(f"{'='*68}")

    # ── Pred vs GT visualisation ──
    n=min(4,len(raw_preds))
    COLORS=['red','blue','green','orange','purple','brown','pink','gray','olive','cyan']
    fig,axes=plt.subplots(2,n,figsize=(5*n,10))
    for col,(img_path,res,gt_boxes,iw,ih) in enumerate(raw_preds[:n]):
        img_obj=Image.open(str(img_path))
        for row,(ax,draw_gt) in enumerate(zip(axes[:,col],[False,True])):
            ax.imshow(img_obj)
            if draw_gt:
                for (gcid,gx1,gy1,gx2,gy2) in gt_boxes:
                    c=COLORS[gcid%len(COLORS)]
                    ax.add_patch(mpatches.Rectangle((gx1,gy1),gx2-gx1,gy2-gy1,lw=1.5,edgecolor=c,facecolor=c,alpha=0.15))
                    ax.add_patch(mpatches.Rectangle((gx1,gy1),gx2-gx1,gy2-gy1,lw=1.5,edgecolor=c,facecolor='none'))
                    cname=CLASS_NAMES[gcid][:8] if gcid<NUM_CLASSES else f'c{gcid}'
                    ax.text(gx1,max(gy1-4,0),cname,color=c,fontsize=5,fontweight='bold',
                            bbox=dict(facecolor='white',alpha=0.6,pad=0))
                ax.set_title(f"GT  n={len(gt_boxes)}",fontsize=8,color='darkgreen')
            else:
                for b in res.boxes:
                    x1,y1,x2,y2=b.xyxy[0].tolist(); cid=int(b.cls[0])
                    c=COLORS[cid%len(COLORS)]
                    ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.5,edgecolor=c,facecolor='none'))
                    ax.text(x1,max(y1-4,0),f"{CLASS_NAMES[cid][:8]}:{float(b.conf[0]):.2f}",
                            color=c,fontsize=5,fontweight='bold',bbox=dict(facecolor='white',alpha=0.6,pad=0))
                r=results_log[col]
                ax.set_title(f"PRED  n={r['num_detections']}  c={r['avg_conf']:.2f}",fontsize=8,color='navy')
            ax.axis('off')
    axes[0][0].set_ylabel("Predictions",fontsize=9,fontweight='bold')
    axes[1][0].set_ylabel("Ground Truth",fontsize=9,fontweight='bold')
    plt.suptitle("D4LA Baseline — Top: Predictions  |  Bottom: Ground Truth\nColour = class",fontsize=10)
    plt.tight_layout()
    vis=PROJECT_ROOT/'output'/'evaluation'/'phase1_d4la_baseline_viz.png'
    plt.savefig(str(vis),dpi=120,bbox_inches='tight'); plt.show()

    # ── Per-class bar chart ──
    cls_sorted=sorted(class_metrics,key=lambda x:class_metrics[x]['f1'])
    f1s=[class_metrics[c]['f1'] for c in cls_sorted]
    precs=[class_metrics[c]['precision'] for c in cls_sorted]
    recs=[class_metrics[c]['recall'] for c in cls_sorted]
    fig,ax=plt.subplots(figsize=(10,max(4,len(cls_sorted)*0.55+2)))
    x=np.arange(len(cls_sorted)); w=0.26
    ax.barh(x-w,precs,w,label='Precision',color='#1f77b4',alpha=0.85)
    ax.barh(x,   recs, w,label='Recall',   color='#2ca02c',alpha=0.85)
    ax.barh(x+w, f1s,  w,label='F1',       color='#d62728',alpha=0.85)
    ax.axvline(0.5,color='gray',linestyle='--',lw=1,alpha=0.5,label='F1=0.5')
    ax.set_yticks(x); ax.set_yticklabels(cls_sorted,fontsize=9)
    ax.set_xlabel('Score',fontsize=11)
    ax.set_title(f"D4LA Per-class P/R/F1  (IoU≥{IOU_THRESH}, conf≥0.25)\n"
                 f"Macro F1={macro_f1:.3f}  |  {len(imgs)} sampled pages  |  "
                 f"Paper mAP@[.5:.95]={CKPT_MAP}%",fontsize=10)
    ax.legend(loc='lower right',fontsize=9); ax.set_xlim(0,1.05); ax.grid(axis='x',alpha=0.3)
    plt.tight_layout()
    bar=PROJECT_ROOT/'output'/'evaluation'/'phase1_d4la_class_metrics.png'
    plt.savefig(str(bar),dpi=120,bbox_inches='tight'); plt.show()
    print(f"Visualisation : {vis}")
    print(f"Bar chart     : {bar}")

    # ── Save JSON ──
    out=PROJECT_ROOT/'output'/'evaluation'/'phase1_english_baseline.json'
    out.write_text(json.dumps({
        "phase":1,
        "dataset":"D4LA test sample (juliozhao/doclayout-yolo-D4LA)",
        "metric_note":"P/R/F1 at IoU≥0.5 on 60 sampled D4LA pages; full mAP on cluster",
        "paper_benchmark":{"mAP_50_95":CKPT_MAP,"source":"Wang et al. 2024 arXiv:2410.12628"},
        "checkpoint_repo":CKPT_REPO_ID,
        "model":CKPT_FILENAME,
        "checkpoint_size_mb":round(CKPT_LOCAL.stat().st_size/1e6,1),
        "num_pages":len(results_log),
        "labeled_pages_pct":round(labeled_pct,1),
        "avg_detections_per_page":round(avg_det,2),
        "avg_confidence":round(avg_conf,3),
        "iou_threshold":IOU_THRESH,
        "macro_precision":round(macro_prec,3),
        "macro_recall":round(macro_rec,3),
        "macro_f1":round(macro_f1,3),
        "baseline_confirmed":ok,
        "per_class_metrics":class_metrics,
        "per_image":results_log,
    },indent=2))
    print(f"Results saved : {out}")
    print("\n✅ Cell 8 complete — proceed to Cell 9 (DocLayNet note) or Cell 11 (Indic pages)")


## Cell 9 — DocLayNet Cluster Note + Proxy Record
**Goal:** DocLayNet (~30 GB download) is too large for Colab's 100 GB Drive limit
alongside D4LA and Indic data. This cell:
1. Prints the exact cluster commands to run DocLayNet evaluation (June 3-8)
2. Writes a clearly-labelled **proxy record** (reuses Cell 8 metrics) so the
   summary JSON in Cell 14 is complete even before the cluster run

**DocLayNet dataset:** `juliozhao/doclayout-yolo-DocLayNet` (HuggingFace dataset)  
**DocLayNet checkpoint for eval:** `juliozhao/DocLayout-YOLO-DocLayNet-Docsynth300K_pretrained`  
**Paper mAP on DocLayNet:** 79.7% (with Docsynth300K pretraining)  
**Target for Phase 1:** ≥ 75% mAP on DocLayNet test split  


In [ ]:
# ── Cell 9: DocLayNet Cluster Note + Proxy Record ──
# DocLayNet (~30 GB) requires cluster storage.
# Source : juliozhao/doclayout-yolo-DocLayNet (HuggingFace dataset)
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

import json

eval_dir   = PROJECT_ROOT / 'output' / 'evaluation'
eng_file   = eval_dir / 'phase1_english_baseline.json'
docln_file = eval_dir / 'phase1_docln_baseline.json'

print("DocLayNet (~30 GB) requires cluster storage.")
print("Colab proxy baseline was run in Cell 8 using 60 D4LA pages.\n")
print("Cluster commands for DocLayNet mAP evaluation (June 3-8):")
print("  # 1. Download DocLayNet dataset (YOLO format, ~30 GB)")
print("  python -c \"")
print("  from huggingface_hub import snapshot_download")
print("  snapshot_download(repo_id='juliozhao/doclayout-yolo-DocLayNet',")
print("                    repo_type='dataset', local_dir='data/raw/DocLayNet')\"")
print()
print("  # 2. Download DocLayNet checkpoint (D4LA ckpt gives ~72%, DocLayNet ckpt gives 79.7%)")
print("  python -c \"")
print("  from huggingface_hub import hf_hub_download")
print("  hf_hub_download('juliozhao/DocLayout-YOLO-DocLayNet-Docsynth300K_pretrained',")
print("                  local_dir='output/checkpoints/')\"")
print()
print("  # 3. Run evaluation")
print("  python val.py --data layout_data/doclaynet/data.yaml \\")
print("                --checkpoint output/checkpoints/<doclaynet_ckpt>.pt \\")
print("                --imgsz 1120  --batch 16")
print()
print("Target: mAP@[.5:.95] ≥ 75% on DocLayNet test split")
print()

if eng_file.exists():
    eng = json.loads(eng_file.read_text())
    docln_record = {
        "phase":1,
        "dataset":"DocLayNet proxy (Cell 8 D4LA metrics reused — not annotated DocLayNet)",
        "metric_note":"PROXY ONLY. Real DocLayNet mAP (target ≥75%) must be run on cluster.",
        "cluster_checkpoint":"juliozhao/DocLayout-YOLO-DocLayNet-Docsynth300K_pretrained",
        "cluster_dataset":"juliozhao/doclayout-yolo-DocLayNet",
        "avg_detections_per_page":eng.get("avg_detections_per_page"),
        "avg_confidence":eng.get("avg_confidence"),
        "baseline_confirmed":eng.get("baseline_confirmed"),
        "target_map":"≥75% mAP@[.5:.95] on real DocLayNet — run on cluster June 3-8",
    }
    docln_file.write_text(json.dumps(docln_record, indent=2))
    print(f"Proxy record written: {docln_file}")
    print(f"  avg_confidence (proxy) : {eng.get('avg_confidence',0):.1%}")
    print()
    print("⚠️  These numbers are from D4LA pages, NOT annotated DocLayNet.")
    print("   Real mAP ≥75% must be verified on the cluster.")
else:
    print("⚠️  Run Cell 8 first to generate the English baseline JSON.")

print("\n✅ Cell 9 complete")


## Cell 10 — Download Indic Document Pages (17 Scripts)
**Goal:** Download real Indic-script document pages to measure the **zero-shot gap**:
how much worse does the English-trained model perform on non-Latin scripts?

**Source:** Wikipedia REST API — `{lang}.wikipedia.org/api/rest_v1/page/pdf/{title}`  
**Why Wikipedia?** Public domain, covers all 17 target scripts, produces realistic
multi-column typeset PDFs (not scanned). Pages are converted to PNG at 2× resolution.

**Scripts covered (17):**
- LTR: Devanagari (Hindi, Marathi), Tamil, Telugu, Bengali, Kannada, Malayalam,
  Gujarati, Odia, Gurmukhi (Punjabi)
- RTL: Urdu Nastaliq, Sindhi

**Note:** These are proxy documents — not the annotated IndicDLP dataset. They give
a visual and confidence-based gap estimate. The real IndicDLP mAP evaluation
(annotated ground truth) runs on the cluster.

**Output:** 4 pages × 17 scripts = ~68 PNG images in `data/raw/IndicDLP/images/test/`  
**Metadata:** `data/raw/IndicDLP/metadata.json` (script family, article, page number)


In [ ]:
# ── Cell 10: Download Real Indic Document Pages ──
# Source : Wikipedia REST API PDF endpoint (public, no auth)
# Format : PDF → PNG @ 2× resolution via PyMuPDF (fitz)
# Scripts: 17 Indic scripts (LTR + RTL)
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

import subprocess, urllib.parse, requests, json as _json, fitz
from PIL import Image
import matplotlib.pyplot as plt
from collections import defaultdict

subprocess.run(['pip', 'install', '-q', 'pymupdf'], check=True)

INDICDLP_DIR = PROJECT_ROOT / 'data' / 'raw' / 'IndicDLP'
IMG_DIR      = INDICDLP_DIR / 'images' / 'test'
IMG_DIR.mkdir(parents=True, exist_ok=True)

# (lang_code, article_title, short_name, script_family)
ARTICLES = [
    ('hi', 'भारत',            'hi_india',      'Devanagari'),
    ('hi', 'हिन्दी साहित्य',   'hi_literature', 'Devanagari'),
    ('ta', 'இந்தியா',          'ta_india',      'Tamil'),
    ('ta', 'தமிழ் இலக்கியம்',  'ta_literature', 'Tamil'),
    ('te', 'భారతదేశం',         'te_india',      'Telugu'),
    ('bn', 'ভারত',            'bn_india',      'Bengali'),
    ('bn', 'বাংলা সাহিত্য',    'bn_literature', 'Bengali'),
    ('kn', 'ಭಾರತ',            'kn_india',      'Kannada'),
    ('ml', 'ഇന്ത്യ',           'ml_india',      'Malayalam'),
    ('gu', 'ભારત',            'gu_india',      'Gujarati'),
    ('mr', 'भारत',            'mr_india',      'Marathi'),
    ('or', 'ଭାରତ',            'or_india',      'Odia'),
    ('ur', 'بھارت',           'ur_india',      'Urdu_RTL'),
    ('ur', 'پاکستان',         'ur_pakistan',   'Urdu_RTL'),
    ('ur', 'اردو زبان',       'ur_language',   'Urdu_RTL'),
    ('pa', 'ਭਾਰਤ',           'pa_india',      'Gurmukhi'),
    ('sd', 'ڀارت',            'sd_india',      'Sindhi_RTL'),
]

PAGES_PER = 4
saved, meta = 0, []
existing_names = {p.name for p in IMG_DIR.glob('*.png')}

print(f"{'Article':<24} {'Script':<16} {'Pages':>5}  Status")
print("─" * 60)

for lang, title, name, family in ARTICLES:
    expected = [f"{name}_p{p:02d}.png" for p in range(PAGES_PER)]
    if all(e in existing_names for e in expected):
        for p in range(PAGES_PER):
            meta.append({"file_name":f"{name}_p{p:02d}.png","script":lang,
                          "family":family,"article":title,"page":p})
            saved += 1
        print(f"  {name:<24} {family:<16} {PAGES_PER:>5}  ✓ (cached)")
        continue

    encoded = urllib.parse.quote(title)
    url     = f"https://{lang}.wikipedia.org/api/rest_v1/page/pdf/{encoded}"
    try:
        resp = requests.get(url, timeout=60, headers={"User-Agent":"DocLayoutYOLO-Research/1.0"})
        resp.raise_for_status()
        doc   = fitz.open(stream=resp.content, filetype="pdf")
        pages = min(PAGES_PER, len(doc))
        for p in range(pages):
            mat = fitz.Matrix(2.0, 2.0)
            pix = doc[p].get_pixmap(matrix=mat, colorspace=fitz.csRGB)
            fname = f"{name}_p{p:02d}.png"
            pix.save(str(IMG_DIR / fname))
            meta.append({"file_name":fname,"script":lang,"family":family,
                          "article":title,"page":p})
            saved += 1
        doc.close()
        print(f"  {name:<24} {family:<16} {pages:>5}  ✓")
    except Exception as e:
        print(f"  {name:<24} {family:<16} {'?':>5}  ✗ {str(e)[:45]}")

(INDICDLP_DIR / 'metadata.json').write_text(
    _json.dumps(meta, indent=2, ensure_ascii=False))

print(f"\nTotal pages saved : {saved}")
print(f"Metadata written  : {INDICDLP_DIR / 'metadata.json'}")

# ── Preview grid ──
by_family = defaultdict(list)
for m in meta:
    by_family[m['family']].append(IMG_DIR / m['file_name'])

families = list(by_family.keys())
MAX_COLS = 4
fig, axes = plt.subplots(len(families), MAX_COLS,
                          figsize=(MAX_COLS*3, len(families)*3.5))
if len(families) == 1: axes = [axes]
for row_idx, family in enumerate(families):
    row_imgs = sorted(by_family[family])[:MAX_COLS]
    for col_idx in range(MAX_COLS):
        ax = axes[row_idx][col_idx]
        if col_idx < len(row_imgs) and row_imgs[col_idx].exists():
            ax.imshow(Image.open(str(row_imgs[col_idx])))
            ax.set_title(row_imgs[col_idx].stem[:16], fontsize=6)
        else:
            ax.set_visible(False)
        ax.axis('off')
    axes[row_idx][0].set_ylabel(family, fontsize=8, fontweight='bold',
                                 rotation=0, labelpad=80, va='center')

plt.suptitle("Indic Document Pages — 17 Scripts (Wikipedia PDFs)\n"
             "Top: LTR scripts  |  Bottom: RTL scripts (Urdu Nastaliq, Sindhi)",
             fontsize=10, y=1.01)
plt.tight_layout()
prev = PROJECT_ROOT / 'output' / 'evaluation' / 'indic_all_scripts_preview.png'
plt.savefig(str(prev), dpi=90, bbox_inches='tight'); plt.show()
print(f"Preview saved : {prev}")
print(f"Total images  : {len(list(IMG_DIR.glob('*.png')))}")
print("\n✅ Cell 10 complete — proceed to Cell 11 (zero-shot evaluation)")


## Cell 11 — Zero-shot Evaluation on Indic Pages ★ SHOWS THE GAP
**Goal:** Run the English-trained DocLayout-YOLO on all 17 Indic scripts and measure
how much performance drops compared to the English (D4LA) baseline from Cell 8.

**This is the motivation for the entire dissertation:**  
English-trained models fail on Indic scripts → we need Phase 2 (synthetic data +
self-training) to close the gap.

**Metrics reported:**
- Avg detections per page and avg confidence per script family
- Gap = English (Cell 8) minus Indic (this cell) for both metrics
- RTL scripts (Urdu, Sindhi) highlighted separately — expected worst failures

**What to expect:**
- LTR scripts (Devanagari, Tamil, etc.): moderate drop in confidence (~5-15%)
- RTL scripts (Urdu Nastaliq, Sindhi): larger drop — model was never trained on
  right-to-left layout or Nastaliq calligraphic style

**Note:** Confidence drop ≠ mAP drop. Actual mAP gap (on annotated IndicDLP) is
typically larger and will be measured on the cluster.

**Results saved to:** `output/evaluation/phase1_indic_zeroshot.json`


In [ ]:
# ── Cell 11: Zero-shot Evaluation on ALL Indic Pages ★ SHOWS THE GAP ──
# Dataset  : Wikipedia Indic pages (proxy — not annotated IndicDLP)
# Baseline : English metrics read from phase1_english_baseline.json (Cell 8)
# Metric   : avg confidence + detections/page per script family
# Gap      : English − Indic  (motivates Phase 2)
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

import subprocess, json
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)
from doclayout_yolo import YOLOv10

IMG_DIR   = PROJECT_ROOT / 'data' / 'raw' / 'IndicDLP' / 'images' / 'test'
META_FILE = PROJECT_ROOT / 'data' / 'raw' / 'IndicDLP' / 'metadata.json'
ENG_FILE  = PROJECT_ROOT / 'output' / 'evaluation' / 'phase1_english_baseline.json'

if not CKPT_LOCAL.exists() and CKPT_DRIVE.exists():
    shutil.copy(CKPT_DRIVE, CKPT_LOCAL)

# ── Load English baseline ──
if ENG_FILE.exists():
    eng_data = json.loads(ENG_FILE.read_text())
    ENG_DET  = eng_data.get('avg_detections_per_page', 12.5)
    ENG_CONF = eng_data.get('avg_confidence', 0.883)
    print(f"English baseline (Cell 8): {ENG_DET:.1f} det/page  {ENG_CONF:.1%} conf")
else:
    ENG_DET, ENG_CONF = 12.5, 0.883
    print("⚠️  English baseline JSON not found — using fallback values. Run Cell 8 first.")

imgs = sorted(IMG_DIR.glob('*.png'))
meta_by_file = {}
if META_FILE.exists():
    for m in json.loads(META_FILE.read_text()):
        meta_by_file[m['file_name']] = m
print(f"Metadata entries : {len(meta_by_file)}")
print(f"Images on disk   : {len(imgs)}")

if not imgs:
    print("❌ No Indic images — run Cell 10 first.")
elif not CKPT_LOCAL.exists():
    print("❌ Checkpoint missing — run Cell 3 first.")
else:
    model  = YOLOv10(str(CKPT_LOCAL))
    COLORS = ['red','blue','green','orange','purple','brown','pink','gray','olive','cyan']

    print(f"\nRunning inference on {len(imgs)} Indic pages (conf≥0.25)...")
    print("─" * 72)

    raw_preds, log_results, per_family = [], [], defaultdict(list)

    for img_path in imgs:
        res   = model.predict(source=str(img_path), imgsz=1024, conf=0.25, verbose=False)[0]
        boxes = res.boxes
        confs = [float(b.conf[0]) for b in boxes]
        avg_c = round(sum(confs)/len(confs), 3) if confs else 0.0
        m     = meta_by_file.get(img_path.name, {})
        family = m.get('family', 'Unknown')

        raw_preds.append((img_path, res, family))
        log_results.append({"file":img_path.name,"family":family,
            "script":m.get('script','?'),"num_detections":len(boxes),"avg_conf":avg_c,
            "detections":[{"class":model.names[int(b.cls[0])],
                           "conf":round(float(b.conf[0]),3)} for b in boxes]})
        per_family[family].append({"n":len(boxes),"conf":avg_c})

    # ── Per-family stats ──
    print(f"\n{'Script Family':<18} {'Pages':>5} {'AvgDet':>8} {'AvgConf':>9}  {'GapDet':>8}  {'GapConf':>9}")
    print("─" * 68)
    family_stats = {}
    for fam, vals in sorted(per_family.items()):
        avg_d = sum(v['n']    for v in vals)/len(vals)
        avg_c = sum(v['conf'] for v in vals)/len(vals)
        gap_d = ENG_DET  - avg_d
        gap_c = ENG_CONF - avg_c
        family_stats[fam] = {"avg_det":avg_d,"avg_conf":avg_c,"gap_det":gap_d,"gap_conf":gap_c}
        flag = "⬅RTL" if "RTL" in fam else ("✅" if avg_d>=6 else "⚠️")
        print(f"  {fam:<16} {len(vals):>5} {avg_d:>8.1f} {avg_c:>8.1%}  "
              f"det:{gap_d:>+6.1f}  conf:{gap_c:>+7.1%}  {flag}")

    overall_det  = sum(r['num_detections'] for r in log_results)/len(log_results)
    overall_conf = sum(r['avg_conf']       for r in log_results)/len(log_results)
    print("─" * 68)
    print(f"  {'OVERALL INDIC':<16} {len(log_results):>5} {overall_det:>8.1f} "
          f"{overall_conf:>8.1%}  det:{ENG_DET-overall_det:>+6.1f}  conf:{ENG_CONF-overall_conf:>+7.1%}")
    print(f"  {'ENGLISH (Cell 8)':<16} {'':>5} {ENG_DET:>8.1f} {ENG_CONF:>8.1%}")

    # ── Full detection grid ──
    COLS=5; ROWS=(len(raw_preds)+COLS-1)//COLS
    fig,axes=plt.subplots(ROWS,COLS,figsize=(COLS*3,ROWS*3.5))
    axes_flat=list(axes.flat) if ROWS>1 else list(axes)
    for i,(img_path,res,family) in enumerate(raw_preds):
        ax=axes_flat[i]; ax.imshow(Image.open(str(img_path)))
        for b in res.boxes:
            x1,y1,x2,y2=b.xyxy[0].tolist(); cid=int(b.cls[0]); conf=float(b.conf[0])
            c=COLORS[cid%len(COLORS)]
            ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.5,edgecolor=c,facecolor='none',alpha=0.85))
            ax.text(x1,max(y1-4,0),f"{model.names[cid][:6]} {conf:.2f}",color=c,fontsize=5,
                    fontweight='bold',bbox=dict(facecolor='white',alpha=0.55,pad=0))
        n_det=len(res.boxes); avg_c_val=round(sum(float(b.conf[0]) for b in res.boxes)/max(n_det,1),2)
        tc='red' if ('RTL' in family or n_det<4) else 'black'
        ax.set_title(f"{family.replace('_RTL','⬅')[:14]}\nn={n_det} c={avg_c_val:.2f}",fontsize=6.5,color=tc)
        ax.axis('off')
    for ax in axes_flat[len(raw_preds):]: ax.axis('off')
    plt.suptitle("Zero-shot on ALL Indic pages (conf≥0.25) — DocLayout-YOLO (English-trained)\n"
                 "Red titles = RTL/failing  |  n=detections  c=avg_confidence",fontsize=9,y=1.005)
    plt.tight_layout()
    vis=PROJECT_ROOT/'output'/'evaluation'/'phase1_indic_all_detections.png'
    plt.savefig(str(vis),dpi=100,bbox_inches='tight'); plt.show()
    print(f"Detection grid : {vis}")

    # ── Gap bar chart ──
    fam_sorted=sorted(family_stats.items(),key=lambda x:x[1]['avg_conf'],reverse=True)
    fam_names=[f[0].replace('_RTL','(RTL)') for f,_ in fam_sorted]
    fam_dets=[s['avg_det']  for _,s in fam_sorted]
    fam_confs=[s['avg_conf'] for _,s in fam_sorted]
    bar_colors=['#d62728' if 'RTL' in f else '#1f77b4' for f in fam_names]

    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,max(5,len(fam_names)*0.5+2)))
    ax1.barh(fam_names,fam_dets,color=bar_colors,alpha=0.8)
    ax1.axvline(ENG_DET,color='green',linestyle='--',lw=2,label=f'English ({ENG_DET:.1f})')
    ax1.set_xlabel('Avg detections/page',fontsize=11); ax1.set_title('Detections: English vs Indic',fontsize=12)
    ax1.legend(); ax1.grid(axis='x',alpha=0.3)
    for i,v in enumerate(fam_dets): ax1.text(v+0.1,i,f'{v:.1f}',va='center',fontsize=9)
    ax2.barh(fam_names,[c*100 for c in fam_confs],color=bar_colors,alpha=0.8)
    ax2.axvline(ENG_CONF*100,color='green',linestyle='--',lw=2,label=f'English ({ENG_CONF:.0%})')
    ax2.set_xlabel('Avg confidence (%)',fontsize=11); ax2.set_title('Confidence: English vs Indic',fontsize=12)
    ax2.legend(); ax2.grid(axis='x',alpha=0.3)
    for i,v in enumerate(fam_confs): ax2.text(v*100+0.3,i,f'{v:.0%}',va='center',fontsize=9)
    plt.suptitle('Model Gap: English (green) vs Indic Scripts\n'
                 'Red = RTL (Urdu/Sindhi worst failures)',fontsize=11)
    plt.tight_layout()
    bar=PROJECT_ROOT/'output'/'evaluation'/'phase1_gap_chart.png'
    plt.savefig(str(bar),dpi=120,bbox_inches='tight'); plt.show()
    print(f"Gap chart      : {bar}")

    # ── Save JSON ──
    out=PROJECT_ROOT/'output'/'evaluation'/'phase1_indic_zeroshot.json'
    out.write_text(json.dumps({
        "phase":1,
        "dataset":"Wikipedia Indic pages (proxy — not annotated IndicDLP)",
        "metric_note":"avg_confidence proxy; annotated IndicDLP mAP requires cluster",
        "source":"Wikipedia REST API — {lang}.wikipedia.org/api/rest_v1/page/pdf/{title}",
        "num_scripts":len(per_family),
        "num_images":len(log_results),
        "avg_detections_per_image":round(overall_det,2),
        "avg_confidence":round(overall_conf,3),
        "english_baseline":{"avg_det":ENG_DET,"avg_conf":ENG_CONF,
                            "source":"phase1_english_baseline.json (Cell 8)"},
        "confidence_gap":round(ENG_CONF-overall_conf,4),
        "detection_gap":round(ENG_DET-overall_det,2),
        "per_family_stats":family_stats,
        "per_image_results":log_results,
    },indent=2,ensure_ascii=False))
    print(f"Results saved  : {out}")
    print("\n✅ Cell 11 complete — proceed to Cell 12 (summary)")


## Cell 12 — Phase 1 Summary Report
**Goal:** Consolidate all Phase 1 results into a single printed report and
`phase1_summary.json`. Reads from Cells 8 and 11 outputs.

Includes:
- English baseline metrics vs Indic zero-shot metrics
- Per-script breakdown sorted by confidence (worst-first)
- Worst-performing script identification
- Pending cluster tasks (real mAP evaluation)
- Thesis statement linking the gap to the Phase 2 motivation

**Output:** `output/evaluation/phase1_summary.json`


In [ ]:
# ── Cell 12: Phase 1 Summary Report ──
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

import json
from datetime import datetime

eval_dir   = PROJECT_ROOT / 'output' / 'evaluation'
eng_file   = eval_dir / 'phase1_english_baseline.json'
indic_file = eval_dir / 'phase1_indic_zeroshot.json'

eng   = json.loads(eng_file.read_text())   if eng_file.exists()   else {}
indic = json.loads(indic_file.read_text()) if indic_file.exists() else {}

ENG_DET  = eng.get('avg_detections_per_page', 'N/A')
ENG_CONF = eng.get('avg_confidence', 'N/A')
IND_DET  = indic.get('avg_detections_per_image', 'N/A')
IND_CONF = indic.get('avg_confidence', 'N/A')
eng_ok   = eng.get('baseline_confirmed', False)
per_family = indic.get('per_family_stats', {})

worst_fam = min(per_family.items(), key=lambda x: x[1]['avg_conf'],
                default=(None,{})) if per_family else (None,{})

det_gap  = round(ENG_DET-IND_DET, 2)   if isinstance(ENG_DET,(int,float))  and isinstance(IND_DET,(int,float))  else None
conf_gap = round(ENG_CONF-IND_CONF, 4) if isinstance(ENG_CONF,(int,float)) and isinstance(IND_CONF,(int,float)) else None

summary = {
    "phase":1, "completed_at":datetime.now().isoformat(),
    "checkpoint_repo": CKPT_REPO_ID,
    "checkpoint_file": CKPT_FILENAME,
    "checkpoint_map_paper": CKPT_MAP,
    "checkpoint_note": CKPT_NOTE,
    "english_proxy_baseline":{
        "dataset":"D4LA test sample (juliozhao/doclayout-yolo-D4LA)",
        "avg_det":ENG_DET,"avg_conf":ENG_CONF,
        "macro_f1":eng.get("macro_f1"),
        "metric_note":"P/R/F1 at IoU≥0.5 on 60 sampled pages — not official mAP",
    },
    "indic_zero_shot":{
        "dataset":"Wikipedia Indic pages — 17 scripts (proxy)",
        "avg_det":IND_DET,"avg_conf":IND_CONF,
        "num_scripts":len(per_family),
        "worst_script":worst_fam[0],
        "worst_conf":worst_fam[1].get('avg_conf') if worst_fam[0] else None,
        "metric_note":"avg_confidence proxy — annotated mAP requires cluster",
    },
    "gaps":{"detection_drop_eng_minus_indic":det_gap,"confidence_drop_eng_minus_indic":conf_gap},
    "phase1_complete":eng_ok,
    "pending_for_cluster":[
        "D4LA full test set mAP evaluation (30,500 images) — target ≥70%",
        "DocLayNet mAP evaluation — target ≥75%",
        "IndicDLP annotated zero-shot mAP — documents the Indic gap quantitatively",
    ],
    "next_notebook":"Phase2_SyntheticData_Colab.ipynb",
}
(eval_dir/'phase1_summary.json').write_text(json.dumps(summary,indent=2))

W=65
print("="*W)
print("  PHASE 1 SUMMARY — DocLayout-YOLO Baseline vs Indic")
print("="*W)
print(f"  Checkpoint : {CKPT_FILENAME}")
print(f"  Repo       : {CKPT_REPO_ID}")
print(f"  Paper mAP  : {CKPT_MAP}%  (D4LA, {CKPT_NOTE})")
print("="*W)
print(f"  {'Metric':<35} {'English':>10} {'Indic':>10} {'Gap':>8}")
print("  "+"─"*(W-2))
if isinstance(ENG_DET,(int,float)) and isinstance(IND_DET,(int,float)):
    print(f"  {'Avg detections/page':<35} {ENG_DET:>10.1f} {IND_DET:>10.1f} {ENG_DET-IND_DET:>+8.1f}")
if isinstance(ENG_CONF,(int,float)) and isinstance(IND_CONF,(int,float)):
    print(f"  {'Avg confidence (proxy)':<35} {ENG_CONF:>10.1%} {IND_CONF:>10.1%} {ENG_CONF-IND_CONF:>+8.1%}")
if eng.get('macro_f1'):
    print(f"  {'Macro F1 @ IoU≥0.5 (English)':<35} {eng['macro_f1']:>10.3f} {'—':>10}")
print()
if per_family:
    print(f"  {'Per-script breakdown':<35} {'Avg Det':>10} {'Avg Conf':>10}")
    print("  "+"─"*(W-2))
    for fam,s in sorted(per_family.items(),key=lambda x:x[1]['avg_conf']):
        rtl=" ⬅RTL" if 'RTL' in fam else ""
        print(f"  {(fam+rtl):<35} {s['avg_det']:>10.1f} {s['avg_conf']:>10.1%}")
print()
print("="*W)
print(f"  Baseline confirmed : {'✅ YES' if eng_ok else '⚠️  Run Cell 8'}")
print(f"  Phase 1 complete   : {'✅ — proceed to Phase 2' if eng_ok else '⚠️  Complete Cells 8 & 11'}")
print("="*W)
print()
print("  PENDING (cluster, June 3-8):")
for item in summary["pending_for_cluster"]:
    print(f"    - {item}")
print()
print("  THESIS STATEMENT:")
if isinstance(ENG_CONF,(int,float)) and isinstance(IND_CONF,(int,float)):
    print(f"  DocLayout-YOLO (English-trained) achieves {ENG_CONF:.0%} avg confidence on English")
    print(f"  documents. On Indic scripts, confidence drops to {IND_CONF:.0%} overall,")
    if worst_fam[0]:
        print(f"  with worst failure on {worst_fam[0]}: {worst_fam[1].get('avg_conf',0):.0%}.")
    print("  This gap motivates Phase 2: synthetic Indic pretraining + self-training.")
print()
print(f"  Summary saved : {eval_dir/'phase1_summary.json'}")
print("  Next         : open Phase2_SyntheticData_Colab.ipynb")
print("\n✅ Cell 12 complete")


## Cell 13 — Backup All Outputs to Drive
**Goal:** Ensure the checkpoint and all evaluation JSONs + charts are safely
backed up to Google Drive before the Colab session ends.

Lists all files in `output/` with sizes for a final health check.


In [ ]:
# ── Cell 13: Backup Checkpoint + Verify Drive Outputs ──
from pathlib import Path
import sys, os, shutil

# ── Re-mount Drive if needed (safe to call multiple times) ──
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=True)

# ── Load shared config (set in Cell 0) ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint constants (set once in Cell 0, reused everywhere)
_cfg_path = PROJECT_ROOT / 'output' / 'checkpoints' / '.cfg'
if _cfg_path.exists():
    import json as _j
    _cfg = _j.loads(_cfg_path.read_text())
    CKPT_REPO_ID  = _cfg['repo_id']
    CKPT_FILENAME = _cfg['filename']
    CKPT_MAP      = _cfg['map']
    CKPT_NOTE     = _cfg['note']
else:
    # Fallback defaults (no-token path)
    CKPT_REPO_ID  = 'juliozhao/DocLayout-YOLO-D4LA-from_scratch'
    CKPT_FILENAME = 'doclayout_yolo_d4la_imgsz1600_from_scratch.pt'
    CKPT_MAP      = 69.8
    CKPT_NOTE     = 'D4LA trained from scratch (no HF token needed)'

CKPT_LOCAL = Path(f'/content/{CKPT_FILENAME}')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / CKPT_FILENAME

if CKPT_LOCAL.exists():
    if not CKPT_DRIVE.exists() or CKPT_DRIVE.stat().st_size != CKPT_LOCAL.stat().st_size:
        shutil.copy(CKPT_LOCAL, CKPT_DRIVE)
        print(f"Checkpoint backed up : {CKPT_DRIVE}  ({CKPT_DRIVE.stat().st_size/1e6:.0f} MB) ✓")
    else:
        print(f"Checkpoint already on Drive : {CKPT_DRIVE.name} ✓")
elif CKPT_DRIVE.exists():
    print(f"Drive checkpoint OK : {CKPT_DRIVE.name} ✓")
else:
    print("❌ Checkpoint not found locally or on Drive. Re-run Cell 3.")

print("\nDrive output/ contents:")
output_dir = PROJECT_ROOT / 'output'
total_size = 0
for f in sorted(output_dir.rglob('*')):
    if f.is_file():
        sz = f.stat().st_size
        total_size += sz
        unit,val = ('MB',sz/1e6) if sz>1e6 else ('KB',sz/1e3)
        print(f"  {str(f.relative_to(output_dir)):<55} {val:6.1f} {unit}")
print(f"\nTotal output size : {total_size/1e6:.1f} MB")
print("\n✅ Cell 13 complete — Phase 1 Colab run finished")


## Phase 1 Completion Checklist

After running all cells, tick these off in `DAILY_TRACKER_JUNE_JULY.md`.

### Colab (done here)
- [ ] Cell 0: GPU runtime confirmed (T4), packages installed, config saved
- [ ] Cell 3: Checkpoint downloaded (~40 MB, D4LA-trained — correct size)
- [ ] Cell 4: Model sanity check passed (detections on synthetic page)
- [ ] Cell 5: D4LA 60-image sample extracted with YOLO-format labels
- [ ] Cell 8: English baseline computed — Macro F1 reported, JSON saved
- [ ] Cell 10: 17 Indic scripts downloaded (68 pages, Wikipedia)
- [ ] Cell 11: Zero-shot gap chart saved (`phase1_gap_chart.png`)
- [ ] Cell 12: `phase1_summary.json` written with thesis statement
- [ ] Cell 13: All outputs backed up to Drive

### Cluster (June 3-8 per EMERGENCY_JUNE_JULY_TIMELINE.md)
- [ ] D4LA full 30,500 test images evaluated → mAP@[.5:.95] ≥ 70%
- [ ] DocLayNet test split evaluated → mAP ≥ 75%
- [ ] IndicDLP annotated zero-shot mAP documented
- [ ] `phase1_d4la_results.json` committed to GitHub

### Key reminders
- The Colab metrics (F1, avg confidence) are **proxy metrics**, not the official mAP
- D4LA checkpoints are legitimately ~40 MB — **do not mistake this for a nano model**
- The `DocStructBench` model (~170 MB) has different class labels and should NOT be
  used for D4LA evaluation
- Real mAP@[.5:.95] on D4LA = 70.3% (Docsynth300K pretrained) / 69.8% (from scratch)
  per Wang et al. 2024 (arXiv:2410.12628)
